## Example 1: Basic Sequence Operations and Chaining

Demonstrate core PyFunctional sequence operations with method chaining and lazy evaluation.

In [ ]:
# Example 1: Basic Sequence Operations and Chaining
from functional import seq

# Create a sequence and demonstrate basic chaining operations
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Chain multiple operations together
result = (seq(data)
    .map(lambda x: x * 2)           # Double each number
    .filter(lambda x: x > 10)       # Keep numbers greater than 10
    .take(5)                        # Take first 5 elements
    .to_list())                     # Convert to list

print(f"Original data: {data}")
print(f"Processed result: {result}")

# Demonstrate lazy evaluation - operations are not executed until to_list()
lazy_seq = (seq(data)
    .map(lambda x: x ** 2)
    .filter(lambda x: x % 2 == 0))

print(f"Lazy sequence created: {type(lazy_seq)}")
print(f"Evaluated result: {lazy_seq.to_list()}")

# Chain reduction operations
sum_result = seq(data).map(lambda x: x ** 2).sum()
max_result = seq(data).filter(lambda x: x % 2 == 0).max()

print(f"Sum of squares: {sum_result}")
print(f"Max even number: {max_result}")
print(f"Chain length: {seq(data).map(lambda x: x * 3).size()}")

## Example 2: Advanced Data Transformations and Grouping

Group records by a key, average each group, and chain filter, map and sort over dictionaries.

In [ ]:
# Example 2: Advanced Data Transformations and Grouping
from functional import seq

# Working with complex data structures
users = [
    {'name': 'Alice', 'age': 28, 'department': 'Engineering', 'salary': 75000},
    {'name': 'Bob', 'age': 32, 'department': 'Marketing', 'salary': 65000},
    {'name': 'Charlie', 'age': 29, 'department': 'Engineering', 'salary': 78000},
    {'name': 'Diana', 'age': 35, 'department': 'Sales', 'salary': 70000},
    {'name': 'Eve', 'age': 26, 'department': 'Engineering', 'salary': 72000}
]

# Group by department and calculate average salary
dept_avg_salary = (seq(users)
    .group_by(lambda u: u['department'])
    .map(lambda item: (item[0], seq(item[1]).map(lambda u: u['salary']).average()))
    .to_dict())

print("Average salary by department:")
for dept, avg_sal in dept_avg_salary.items():
    print(f"  {dept}: ${avg_sal:,.2f}")

# Complex transformation pipeline
senior_eng_salaries = (seq(users)
    .filter(lambda u: u['department'] == 'Engineering')
    .filter(lambda u: u['age'] >= 28)
    .map(lambda u: {'name': u['name'], 'adjusted_salary': u['salary'] * 1.1})
    .sorted(key=lambda u: u['adjusted_salary'], reverse=True)
    .to_list())

print(f"\nSenior Engineering salaries (with 10% adjustment):")
for emp in senior_eng_salaries:
    print(f"  {emp['name']}: ${emp['adjusted_salary']:,.2f}")

# Statistical operations
engineering_ages = (seq(users)
    .filter(lambda u: u['department'] == 'Engineering')
    .map(lambda u: u['age']))

print(f"\nEngineering department age stats:")
print(f"  Average age: {engineering_ages.average():.1f}")
print(f"  Min age: {engineering_ages.min()}")
print(f"  Max age: {engineering_ages.max()}")

## Example 3: Parallel Processing with pseq

Run the same pipeline with `seq` and with the multiprocessing `pseq`, and check that both give the same results.

In [ ]:
# Example 3: Parallel Processing with pseq  
from functional import seq, pseq
import time

# Generate sample data for parallel processing
numbers = list(range(1, 1001))  # 1000 numbers

def expensive_operation(x):
    """Simulate expensive computation"""
    import time  # Import inside function for parallel processing
    time.sleep(0.001)  # Small delay to simulate work
    return x ** 2 + x ** 0.5

# Sequential processing with regular seq
start_time = time.time()
sequential_result = (seq(numbers[:100])  # Use smaller subset for demo
    .map(expensive_operation)
    .filter(lambda x: x > 50)
    .take(20)
    .to_list())
sequential_time = time.time() - start_time

# Parallel processing with pseq
start_time = time.time()  
parallel_result = (pseq(numbers[:100])   # Use smaller subset for demo
    .map(expensive_operation)
    .filter(lambda x: x > 50)
    .take(20)
    .to_list())
parallel_time = time.time() - start_time

print(f"Sequential processing time: {sequential_time:.4f} seconds")
print(f"Parallel processing time: {parallel_time:.4f} seconds")
print(f"Results match: {sequential_result == parallel_result}")
assert len(sequential_result) == 20, sequential_result
assert sequential_result == parallel_result, (sequential_result, parallel_result)
print(f"First 5 results: {parallel_result[:5]}")

# Parallel aggregation operations
large_dataset = list(range(1, 10001))
parallel_sum = pseq(large_dataset).map(lambda x: x ** 2).sum()
parallel_avg = pseq(large_dataset).map(lambda x: x ** 2).average()

print(f"\nParallel aggregation on 10,000 numbers:")
print(f"Sum of squares: {parallel_sum:,}")
print(f"Average of squares: {parallel_avg:,.2f}")
# Sum of the first n squares is n(n+1)(2n+1)/6.
assert parallel_sum == 10000 * 10001 * 20001 // 6, parallel_sum
assert abs(parallel_avg - parallel_sum / 10000) < 1e-6, parallel_avg

# Parallel text processing example
words = ['hello', 'world', 'python', 'functional', 'programming'] * 50
processed_words = (pseq(words)
    .map(lambda w: w.upper())
    .filter(lambda w: len(w) > 5)
    .distinct()
    .to_list())

print(f"\nProcessed {len(words)} words, found {len(processed_words)} unique long words")
assert sorted(processed_words) == ["FUNCTIONAL", "PROGRAMMING", "PYTHON"], processed_words
print(f"Sample results: {processed_words[:5]}")

## Example 4: File I/O and Stream Processing

Read a CSV file with `seq.open`, process a generator lazily, and count word frequencies.

In [ ]:
# Example 4: File I/O and Stream Processing
from functional import seq
import tempfile
import os

# Create sample data files for processing
sample_data = [
    "apple,fruit,red,2.50",
    "banana,fruit,yellow,1.20", 
    "carrot,vegetable,orange,0.80",
    "broccoli,vegetable,green,2.10",
    "grape,fruit,purple,3.00",
    "lettuce,vegetable,green,1.50"
]

# Create temporary files for demonstration
temp_dir = tempfile.mkdtemp()
csv_file = os.path.join(temp_dir, "products.csv")

with open(csv_file, 'w') as f:
    f.write("name,category,color,price\n")
    for line in sample_data:
        f.write(line + "\n")

# Process file with PyFunctional
processed_data = (seq.open(csv_file)
    .drop(1)  # Skip header
    .map(lambda line: line.strip().split(','))
    .map(lambda fields: {
        'name': fields[0],
        'category': fields[1], 
        'color': fields[2],
        'price': float(fields[3])
    })
    .filter(lambda item: item['category'] == 'fruit')
    .sorted(key=lambda item: item['price'], reverse=True)
    .to_list())

print("Fruit products sorted by price (highest first):")
for item in processed_data:
    print(f"  {item['name']}: ${item['price']:.2f} ({item['color']})")

# Stream processing with generators
def number_generator():
    """Generate infinite sequence of numbers"""
    i = 1
    while i <= 1000:  # Limit for demo
        yield i
        i += 1

# Process infinite stream lazily
stream_result = (seq(number_generator())
    .map(lambda x: x ** 2)
    .filter(lambda x: x % 17 == 0)  # Numbers divisible by 17
    .take(10)
    .to_list())

print(f"\nFirst 10 perfect squares divisible by 17: {stream_result}")

# Clean up temporary files
os.unlink(csv_file)
os.rmdir(temp_dir)

# Text processing pipeline
text_data = "The quick brown fox jumps over the lazy dog " * 10
word_analysis = (seq(text_data.split())
    .map(lambda w: w.lower().strip('.,!?'))
    .group_by(lambda w: w)
    .map(lambda item: (item[0], len(item[1])))
    .sorted(key=lambda item: item[1], reverse=True)
    .take(5)
    .to_list())

print(f"\nTop 5 most frequent words:")
for word, count in word_analysis:
    print(f"  '{word}': {count} times")

## Example 5: Production Data Pipeline and Analytics

Group and summarise generated log records with `seq` and `pseq`.

In [ ]:
# Example 5: Production Data Pipeline and Analytics
from functional import seq, pseq
import json
import random

# Simulate production log data
def generate_log_entries(count=1000):
    """Generate sample log entries for processing"""
    services = ['api', 'database', 'cache', 'auth', 'frontend']
    levels = ['INFO', 'WARN', 'ERROR', 'DEBUG']
    
    for i in range(count):
        yield {
            'timestamp': f"2024-01-{(i % 31) + 1:02d}T{(i % 24):02d}:00:00Z",
            'service': random.choice(services),
            'level': random.choice(levels),
            'response_time': random.uniform(50, 2000),
            'user_id': f"user_{random.randint(1, 100)}"
        }

# Production analytics pipeline
log_data = list(generate_log_entries(500))  # Generate sample data

# Error analysis pipeline
error_analysis = (seq(log_data)
    .filter(lambda log: log['level'] == 'ERROR')
    .group_by(lambda log: log['service'])
    .map(lambda item: {
        'service': item[0],
        'error_count': len(item[1]),
        'avg_response_time': seq(item[1]).map(lambda l: l['response_time']).average()
    })
    .sorted(key=lambda item: item['error_count'], reverse=True)
    .to_list())

print("Error Analysis by Service:")
for analysis in error_analysis:
    print(f"  {analysis['service']}: {analysis['error_count']} errors, "
          f"avg response: {analysis['avg_response_time']:.1f}ms")

# Performance monitoring pipeline  
slow_requests = (seq(log_data)
    .filter(lambda log: log['response_time'] > 1000)
    .group_by(lambda log: log['service'])
    .map(lambda item: {
        'service': item[0],
        'slow_count': len(item[1]),
        'max_response_time': seq(item[1]).map(lambda l: l['response_time']).max()
    })
    .filter(lambda item: item['slow_count'] > 0)
    .to_list())

print(f"\nSlow Request Analysis (>1000ms):")
for service in slow_requests:
    print(f"  {service['service']}: {service['slow_count']} slow requests, "
          f"slowest: {service['max_response_time']:.1f}ms")

# User activity analytics with parallel processing
user_activity = (pseq(log_data)
    .group_by(lambda log: log['user_id'])
    .map(lambda item: {
        'user_id': item[0],
        'total_requests': len(item[1]),
        'error_rate': len([l for l in item[1] if l['level'] == 'ERROR']) / len(item[1]),
        'avg_response': sum(l['response_time'] for l in item[1]) / len(item[1])
    })
    .filter(lambda user: user['total_requests'] >= 5)  # Active users only
    .sorted(key=lambda user: user['error_rate'], reverse=True)
    .take(5)
    .to_list())

print(f"\nTop 5 Users by Error Rate (min 5 requests):")
for user in user_activity:
    print(f"  {user['user_id']}: {user['error_rate']:.1%} error rate, "
          f"{user['total_requests']} requests, {user['avg_response']:.1f}ms avg")